# Workshop 3 · 03 — CV-Teaser: Wagon-Schaden → Werkstatt

Der **unstrukturierteste** Datentyp: ein **Bild**. Eine **multimodale** AI Function bekommt den Foto-Pfad und liefert pro Wagon einen strukturierten Schadensbericht (Typ, Schweregrad, Empfehlung, UIC per OCR). Daraus leiten wir den **Werkstatt-Auftrag** ab.

Referenz: https://learn.microsoft.com/fabric/data-science/ai-functions/multimodal-overview

In [ ]:
import synapse.ml.spark.aifunc as aifunc

# OneLake-ABFSS-Pfad zum Lakehouse bauen
onelake_endpoint = notebookutils.conf.get("trident.onelake.endpoint").replace("https://", "")
workspace_id = notebookutils.runtime.context['currentWorkspaceId']
lakehouse_id = notebookutils.runtime.context['defaultLakehouseId']

base_path = f"abfss://{workspace_id}@{onelake_endpoint}/{lakehouse_id}/Files/wagon_photos_w3"

# Alle Wagon-Fotos als Pfad-Spalte laden
files_df = aifunc.list_file_paths(base_path)
display(files_df)

In [ ]:
# This code uses AI. Always review output for mistakes.

report_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "schadensbericht",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "damage_type": {"type": "string",
                                "enum": ["rost", "kratzer", "delle", "graffiti", "riss", "kein_schaden"]},
                "severity": {"type": "integer"},
                "recommended_action": {"type": "string"},
                "uic_number": {"type": "string"},
                "safety_relevant": {"type": "boolean"},
            },
            "required": ["damage_type", "severity", "recommended_action", "uic_number", "safety_relevant"],
            "additionalProperties": False,
        },
    },
}

prompt = ("Du bist Pruefingenieur fuer Schienenfahrzeuge. Analysiere das Foto eines Waggon-Panels. "
          "Bestimme Schadenstyp, Schweregrad 1 (kosmetisch) bis 5 (sicherheitskritisch), eine knappe "
          "Handlungsempfehlung auf Deutsch, lies die 12-stellige UIC-Nummer (sonst 'unbekannt') und ob "
          "der Schaden sicherheitsrelevant ist.")

reports = files_df.ai.generate_response(
    prompt=prompt,
    col_types={"file_path": "path"},
    response_format=report_schema,
    output_col="report_json",
)
display(reports.select("file_path", "report_json"))

In [ ]:
from pyspark.sql.functions import from_json, col, when, lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType

rschema = StructType([
    StructField("damage_type", StringType()),
    StructField("severity", IntegerType()),
    StructField("recommended_action", StringType()),
    StructField("uic_number", StringType()),
    StructField("safety_relevant", BooleanType()),
])

findings = (reports
            .withColumn("r", from_json(col("report_json"), rschema))
            .select(col("file_path").cast("string").alias("image_path"),
                    "r.damage_type", "r.severity", "r.recommended_action",
                    "r.uic_number", "r.safety_relevant"))

work = (findings
        .withColumn("workshop", when(col("damage_type").isin("rost", "kratzer"), lit("Karosserie/Lack"))
                    .when(col("damage_type") == "graffiti", lit("Reinigung"))
                    .when(col("damage_type").isin("delle", "riss"), lit("Strukturpruefung"))
                    .otherwise(lit("-")))
        .withColumn("dispatch", (col("severity") >= 4) | (col("safety_relevant") == True))
        .withColumn("created_at", current_timestamp()))

work.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("schaden_work_orders")
display(work.orderBy(col("severity").desc()))

### Takeaway & Operationalisierung

Aus Bildern wurde die strukturierte Tabelle `schaden_work_orders` mit Werkstatt-Zuordnung und `dispatch`-Flag. Der letzte Schritt im echten Workload:
- **Activator**-Regel auf `dispatch = true` → Teams-Nachricht an die Werkstatt-Disposition.
- **Power BI** (Direct Lake) Dashboard über offene Aufträge.

> Tiefer Computer-Vision-Use-Case (eigenes ONNX-Modell, BYOM, OCR) → **Workshop 1**.